# LizyML Tutorial: Codegen Export

This notebook demonstrates **`Model.export_code()`** — LizyML's code generation feature
that produces a fully self-contained training and inference pipeline with no LizyML dependency.

**What codegen does:**
- Generates `train.py` — reproduces the full training pipeline (preprocessing + LightGBM CV)
- Generates `predict.py` — loads artifacts and runs inference on new data
- Generates `test_equivalence.py` — verifies that generated code matches LizyML's OOF predictions
- Generates `config.json` — all parameters in plain JSON
- Generates `requirements.txt` — minimal dependency list
- Copies `artifacts/` — serialised pipeline state (encoders, boosters, calibrators)

**Generated code only requires:** `lightgbm`, `numpy`, `pandas`, `scikit-learn`

**Contents:**
1. Setup
2. Data
3. Config
4. Model Fit
5. Export Code
6. Inspect Generated Files
7. Running Inference with Generated Code
8. Equivalence Test

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import subprocess
import tempfile
from pathlib import Path

import pandas as pd
from sklearn.datasets import make_regression

from lizyml import Model

# Temporary output directory — replace with a real path to persist generated files
TMP_DIR = Path(tempfile.mkdtemp()) / "codegen_demo"
print(f"Output directory: {TMP_DIR}")

## 2. Data

Synthetic regression dataset with 500 samples and 5 features.
Codegen works for all three tasks (regression, binary, multiclass) —
regression is the simplest to demonstrate as it has no calibration step.

In [ ]:
X, y = make_regression(
    n_samples=500,
    n_features=5,
    n_informative=4,
    noise=10.0,
    random_state=42,
)

feature_names = [f"feature_{i:02d}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print(f"Shape: {df.shape}")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}]")
df.head()

## 3. Config

Standard regression config with 3-fold KFold.
Codegen captures the entire pipeline configuration and is not sensitive
to which split method or parameters you use.

In [ ]:
config = {
    "config_version": 1,
    "task": "regression",
    "data": {
        "target": "target",
    },
    "model": {
        "name": "lgbm",
        "params": {
            "objective": "regression",
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 4,
            "feature_fraction": 0.8,
        },
    },
    "split": {
        "method": "kfold",
        "n_splits": 3,
    },
    "training": {
        "seed": 42,
    },
    "evaluation": {
        "metrics": ["rmse", "mae", "r2"],
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
model.fit(data=df)
print("Fit complete.")

In [ ]:
model.evaluate_table().round(4)

## 5. Export Code

`model.export_code(path)` writes all generated files to the specified directory.
The directory is created if it does not exist.

In [ ]:
model.export_code(TMP_DIR)
print(f"Exported to: {TMP_DIR}")

## 6. Inspect Generated Files

The export directory contains everything needed to reproduce training
and run inference in a LizyML-free environment.

In [ ]:
# List all generated files
all_files = sorted(TMP_DIR.rglob("*"))
for f in all_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        rel = f.relative_to(TMP_DIR)
        print(f"  {rel}  ({size_kb:.1f} KB)")

In [ ]:
# Display config.json — all training parameters in plain JSON
config_path = TMP_DIR / "config.json"
with open(config_path) as f:
    generated_config = json.load(f)

print(json.dumps(generated_config, indent=2))

In [ ]:
# Display requirements.txt
req_path = TMP_DIR / "requirements.txt"
print(req_path.read_text())

## 7. Running Inference with Generated Code

The generated `predict.py` accepts a CSV file and outputs predictions.
It loads the serialised artifacts from the `artifacts/` directory.

**Usage (in a LizyML-free environment):**
```bash
# Install only the required dependencies
pip install lightgbm numpy pandas scikit-learn

# Run predictions on new data
python predict.py test.csv -o predictions.csv
```

The `predict.py` script:
- Loads `artifacts/` (booster files + encoder state)
- Applies the same feature transformations used during training
- Averages predictions across all CV folds
- Writes output to the specified file or stdout

In [ ]:
# Demonstrate that predict.py can be called directly from Python
X_new = df.drop(columns=["target"]).head(5)
test_csv_path = TMP_DIR / "test_input.csv"
X_new.to_csv(test_csv_path, index=False)

result = subprocess.run(
    ["python", str(TMP_DIR / "predict.py"), str(test_csv_path)],
    capture_output=True,
    text=True,
)
print("stdout:", result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr)

## 8. Equivalence Test

The generated `test_equivalence.py` verifies that the generated `predict.py`
produces predictions that match LizyML's OOF predictions within a tight tolerance.
This is the key correctness guarantee of the export.

**Usage:**
```bash
pytest test_equivalence.py -v
```

The equivalence test checks:
- Prediction shape matches
- Values match within `atol=1e-5` (accounting for floating point differences)
- No NaN or Inf values in output

In [ ]:
# Run the equivalence test against the training data
eq_path = TMP_DIR / "test_equivalence.py"
if eq_path.exists():
    result = subprocess.run(
        ["python", "-m", "pytest", str(eq_path), "-v", "--tb=short"],
        capture_output=True,
        text=True,
        cwd=TMP_DIR,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
else:
    print("test_equivalence.py not found in export directory")